# Extract Data from Processed Files

Extract and organize data into task-specific categories:
- Power data from all optimizer channels
- Microclimate data (temperature, humidity, wind, irradiance and soil moisture)
- Weather data (meteorological conditions)

**Data Source Priority**: Uses interpolated data if available, otherwise uses downsampled data, then quality controlled data.

In [ ]:
import sys
from pathlib import Path
import warnings
import importlib
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path('../src').resolve()))

# Import and force reload to pick up changes
import extract_data
importlib.reload(extract_data)
from extract_data import DataExtractor, list_available_tasks, split_files_by_date, split_columns_by_date

## Configuration

In [ ]:
# Display configuration from config file
extractor = DataExtractor()

print("="*80)
print("Configuration:")
print(f"  Input directory: {extractor.get_input_directory()}")
print(f"  Output directory: {extractor.output_dir}")
print(f"  Global date range: {extractor.start_date} to {extractor.end_date}")
print(f"  Note: Soil moisture uses custom range (2024-04-01 to 2024-08-31)")
print(f"\n  Available tasks: {list_available_tasks()}")
print("="*80)

## Extract Power Data

In [ ]:
# Extract power data (uses config file settings automatically)
extractor = DataExtractor('power')
power_results = extractor.extract()

print(f"Power extraction complete: {len(power_results)} datasets extracted")
for result in power_results:
    print(f"  {result['output_file'].name}: {result['records']} records")
print("="*80)

## Extract Microclimate Data

In [ ]:
# Extract microclimate data
extractor = DataExtractor('microclimate')
microclimate_results = extractor.extract()

print(f"Microclimate extraction complete: {len(microclimate_results)} datasets extracted")
for result in microclimate_results:
    records_info = f"{result['records']} records"
    if 'SoilMoisture' in str(result['output_file']):
        records_info += " (Apr-Aug 2024)"
    print(f"  {result['output_file'].name}: {records_info}")
print("="*80)

## Extract Weather Data

In [ ]:
# Extract weather data
extractor = DataExtractor('weather')
weather_results = extractor.extract()

print(f"Weather extraction complete: {len(weather_results)} datasets extracted")
for result in weather_results:
    print(f"  {result['output_file'].name}: {result['records']} records")
print("="*80)

## Preliminary Missing Rate Analysis

In [ ]:
import os
import pandas as pd
import importlib
import missing_rate_analysis
importlib.reload(missing_rate_analysis)
from missing_rate_analysis import calculate_task_missing_rate

print("Executing Preliminary Missing Rate Analysis")

# Get the extracted directory
if 'PIPELINE_OUTPUT_ROOT' in os.environ:
    extracted_dir = Path(os.environ['PIPELINE_OUTPUT_ROOT']) / 'extracted'
else:
    extracted_dir = extractor.output_dir

if extracted_dir.exists():
    # For missing rate analysis, we analyze the folders separately since they're in different directories
    data_types = ['power', 'microclimate', 'weather']
    missing_summary = []
    
    for data_type in data_types:
        if data_type == 'power':
            time_range = (6,18)
        else:
            time_range = None
        result = calculate_task_missing_rate(extracted_dir, data_type, time_range=time_range)
        if result:
            missing_summary.append(result)
    
    if missing_summary:
        summary_df = pd.DataFrame(missing_summary)
        print("\nMissing Rate Summary:")
        print(summary_df.to_string(index=False))
    else:
        print("No data found for missing rate analysis")
else:
    print(f"Extracted directory not found: {extracted_dir}")

print("="*80)

## [Special Process] Split Wedelia Trilobata Files by Time Period

Search for files with the name that contains Wedelia trilobata

In [ ]:
# Define split parameters
import os

# Use pipeline output directory if available, otherwise use default
if 'PIPELINE_OUTPUT_ROOT' in os.environ:
    extracted_dir = Path(os.environ['PIPELINE_OUTPUT_ROOT']) / 'extracted'
else:
    extracted_dir = Path('../data/extracted')

split_date = '2024-12-01'  # Adjust this date as needed

print(f"Looking for Wedelia trilobata files in: {extracted_dir}")

if not extracted_dir.exists():
    print(f"WARNING: Extracted directory not found: {extracted_dir}")
    print("Skipping file splitting...")
else:
    # Split Wedelia trilobata files using the utility function
    results = split_files_by_date(
        input_directory=extracted_dir,
        split_date=split_date,
        file_pattern='*WedTri*.csv',  # Adjust pattern to match your files
        part1_suffix='_WT',
        part2_suffix='_WT-to-Sed',
        recursive=True,  # Search subdirectories
        remove_original=True  # Remove original files after splitting
    )
    
    print(f"\nSplit {len(results)} Wedelia trilobata files:")
    for result in results:
        print(f"\n  {result['original_file'].name}")
        print(f"    Part 1 (before {split_date}): {result['part1_records']} records")    
        print(f"    Part 2 (from {split_date}): {result['part2_records']} records")

Search for DataFrame columns that contains Wedelia trilobata

In [ ]:
# Split columns in power and microclimate files where Wedelia trilobata was replaced with Sedum
import os

# Use pipeline output directory if available, otherwise use default
if 'PIPELINE_OUTPUT_ROOT' in os.environ:
    extracted_dir = Path(os.environ['PIPELINE_OUTPUT_ROOT']) / 'extracted'
else:
    extracted_dir = Path('../data/extracted')

split_date = '2024-12-01'  # Date when Wedelia trilobata was replaced with Sedum

print(f"Looking for files to split columns in: {extracted_dir}")

files_to_split = [
    'power/PV_Power.csv'
]

if not extracted_dir.exists():
    print(f"WARNING: Extracted directory not found: {extracted_dir}")
    print("Skipping column splitting...")
else:
    # Split columns matching 'WedTri' or 'WT' patterns
    results = []
    for file_rel_path in files_to_split:
        file_path = extracted_dir / file_rel_path
        if file_path.exists():
            result = split_columns_by_date(
                file_path=file_path,
                split_date=split_date,
                column_patterns=['WedTri', 'WT'],
                part1_suffix='_WT',
                part2_suffix='_WT-to-Sed',
                keep_original=False  # Remove original columns
            )
            if result:
                results.append(result)
                print(f"\n{file_path.name}:")
                print(f"  Split {len(result['split_columns'])} columns: {', '.join(result['split_columns'])}")
                print(f"  Part 1 (before {split_date}): {result['part1_records']} records")
                print(f"  Part 2 (from {split_date}): {result['part2_records']} records")
        else:
            print(f"\nFile not found: {file_path}")
    
    print(f"\n\nProcessed {len(results)} files successfully")

## Summary

In [ ]:
# Calculate total files extracted
total_files = len(power_results) + len(microclimate_results) + len(weather_results)

print(f"{'='*60}")
print(f"Extraction complete: {total_files} datasets ready for analysis")
print(f"  Power: {len(power_results)} files")
print(f"  Microclimate: {len(microclimate_results)} files")
print(f"  Weather: {len(weather_results)} files")
print(f"Output directory: {extractor.output_dir}")

# Display info about data source
if extractor._check_interpolation_enabled():
    print("Note: Using interpolated data (contains synthetic values)")
    print("Check interpolation_report.json for details")
else:
    print("Using downsampled data (measured values only)")

# Display date range info
print(f"\nDate ranges:")
print(f"  Power & Microclimate: {extractor.start_date} to {extractor.end_date}")
print(f"  Soil Moisture: 2024-04-01 to 2024-08-31 (per-file override)")
print(f"{'='*60}")